# Qanuni Acceptance Pack

هذا الـ notebook هو مسار قبول منظم للنسخة الحرة من Qanuni.
الفكرة أنه يختبر النسخة كما يراها المستخدم: أدوات أساسية، أدوات ذرية، workflow، agent، observability، وأمثلة أخطاء واضحة.


## قبل البدء

داخل Jupyter يفضل أولًا:

```python
%pip install --upgrade qanuni-sdk
```

ثم أعد تشغيل الـ kernel وتأكد من النسخة والمسار.
إذا كنت تختبر من داخل هذا المستودع نفسه فالأفضل أن تبقي `new.ipynb` للتجارب الحرة، وتستخدم هذا الملف كمرجع قبول ثابت.


In [ ]:
import qanuni

print(qanuni.__version__)
print(qanuni.__file__)


In [ ]:
from qanuni.acceptance import (
    build_acceptance_client,
    list_sample_documents,
    load_sample_document,
    sample_document_path,
)

sample_documents = list_sample_documents()
sample_documents


In [ ]:
from pathlib import Path
from tempfile import mkdtemp

working_dir = Path(mkdtemp(prefix="qanuni-acceptance-notebook-"))
client = build_acceptance_client(
    mode="mocked",
    observability_persist=True,
    working_dir=working_dir,
)

print("Working directory:", working_dir)
print("First sample path:", sample_document_path(sample_documents[0]))


In [ ]:
end_of_service = client.labor.end_of_service(
    monthly_salary=12000,
    years_of_service=7.5,
    termination_reason="resignation",
    contract_type="indefinite",
)
probation = client.labor.probation_check(
    probation_days=120,
    extension_in_writing=False,
    contract_type="indefinite",
)

print(end_of_service.total_amount)
print(probation.is_legal)
print(probation.violations)


In [ ]:
service_agreement_text = load_sample_document("service_agreement_ar.md")

classification = client.legal.classify_document_type(
    document_text=service_agreement_text,
    document_type="service_agreement",
)
clauses = client.legal.extract_clauses(
    document_text=service_agreement_text,
    document_type="service_agreement",
)
parties = client.legal.extract_parties(
    document_text=service_agreement_text,
    document_type="service_agreement",
)
obligations = client.legal.extract_obligations(
    document_text=service_agreement_text,
    document_type="service_agreement",
)

print(classification.primary_document_type)
print([party.name for party in parties.parties])
print([clause.clause_type for clause in clauses.clauses])
print([obligation.action for obligation in obligations.obligations])


In [ ]:
contract_review = client.workflow.contract_review(
    document_text=service_agreement_text,
    contract_type="service_agreement",
    include_redlines=True,
)

print(contract_review.executive_summary)
print(contract_review.risk_level)
print(contract_review.amendment_recommendations)


In [ ]:
employment_contract_text = load_sample_document("employment_contract_ar.md")

employment_review = client.workflow.employment_review(
    document_text=employment_contract_text,
    document_type="employment_contract",
    contract_type="indefinite",
    probation_days=120,
    extension_in_writing=False,
    monthly_salary=10000,
    years_of_service=2,
    termination_reason="termination_by_employer",
)

print(employment_review.executive_summary)
print(employment_review.employment_risks)


In [ ]:
from qanuni.agent.models import AgentScenario

agent_result = client.agent.run(
    goal="أريد مراجعة العقد ثم تجهيز مسودة مطالبة قبل النزاع.",
    scenario_hint=AgentScenario.CONTRACT_DISPUTE_NOTICE,
    documents=[
        {
            "name": "عقد خدمات",
            "text": service_agreement_text,
            "document_type": "service_agreement",
            "role": "primary",
        },
        {
            "name": "مستند داعم للمطالبة",
            "text": load_sample_document("prelitigation_support_ar.md"),
            "document_type": "demand_support",
            "role": "supporting",
        },
    ],
    facts={
        "sender_name": "شركة ألف",
        "recipient_name": "شركة باء",
        "claim_type": "مستحقات تعاقدية",
        "claim_amount": 85000,
        "incident_description": "تأخر في سداد مستحقات عقد خدمات تقنية.",
        "deadline_days": 7,
        "threat_of_action": "سيتم اتخاذ الإجراءات القانونية المناسبة عند عدم السداد.",
    },
)

print(agent_result.status)
print(agent_result.answer_text)


In [ ]:
client.observability.clear()

first_risk = client.contracts.risk_score(
    contract_text=service_agreement_text,
    contract_type="service_agreement",
)
second_risk = client.contracts.risk_score(
    contract_text=service_agreement_text,
    contract_type="service_agreement",
)

events = [event.model_dump(mode="json") for event in client.observability.snapshot()]
print("Event count:", len(events))
print("Cache statuses:", [event.get("cache_status") for event in events])

observability_log_path = working_dir / ".qanuni_observability" / "events.jsonl"
print(observability_log_path)
if observability_log_path.exists():
    print(observability_log_path.read_text(encoding="utf-8")[:800])


In [ ]:
from qanuni.core.exceptions import QanuniError

try:
    client.contracts.gap_analysis(contract_type="service_agreement")
except QanuniError as error:
    print(error.error_code)
    print(error)
    print(error.details)


## Live reruns and MCP smoke

عندما تريد اختبار المسار الحقيقي مع OpenAI:

```python
import os
from qanuni import LegalClient

live_client = LegalClient(api_key=os.getenv("OPENAI_API_KEY"))
```

وللتحقق من السطح الخارجي عبر MCP بعد تثبيت extra المناسب:

```bash
qanuni-mcp-smoke --mode mocked --working-dir .qanuni_acceptance
qanuni-mcp-smoke --mode live --working-dir .qanuni_acceptance
```

ولتشغيل acceptance pack نفسه من الـ CLI:

```bash
qanuni-acceptance --mode mocked --persist-observability --working-dir .qanuni_acceptance
```
